# Hybrid AHP-TOPSIS Multi-Criteria Decision Making for Wave Energy Site Selection

---

## Overview

Selecting an optimal site for wave energy converter (WEC) deployment requires balancing multiple, often conflicting, objectives. This notebook implements a **hybrid MCDM** (Multi-Criteria Decision Making) framework that combines two classical methods:

1. **AHP** (Analytic Hierarchy Process) — determines the **weight** of each criterion through pairwise expert comparisons (Saaty, 1980)
2. **TOPSIS** (Technique for Order of Preference by Similarity to Ideal Solution) — **ranks** candidate sites by their geometric distance to the ideal and anti-ideal alternatives

### Why AHP-TOPSIS?

| Method | Strength | Limitation |
|--------|----------|------------|
| AHP | Intuitive pairwise comparisons; consistency check via CR | Only provides weights, not a ranking of alternatives |
| TOPSIS | Objective ranking based on distance to ideal | Requires externally supplied weights |

The hybrid approach leverages AHP's strength in weight elicitation and TOPSIS's strength in alternative ranking.

### Criteria for WEC Site Selection

| # | Criterion | Type | Rationale |
|---|-----------|------|-----------|
| 1 | Wave Power (kW/m) | Benefit | Higher energy flux = more electricity |
| 2 | Energy Consistency (%) | Benefit | Stable resource reduces intermittency |
| 3 | Distance to Shore (km) | Cost | Longer cables increase CAPEX |
| 4 | Distance to Grid (km) | Cost | Farther from grid = higher connection cost |
| 5 | Environmental Sensitivity (1-5) | Cost | Higher score = more ecological constraints |

### Dependencies

```
numpy
```

In [ ]:
import numpy as np

## 1. AHP: Analytic Hierarchy Process

### Theory

AHP uses a square **pairwise comparison matrix** $A$ where entry $a_{ij}$ indicates how much more important criterion $i$ is relative to criterion $j$, using Saaty's 1-9 scale:

| Value | Interpretation |
|-------|---------------|
| 1 | Equal importance |
| 3 | Moderate importance |
| 5 | Strong importance |
| 7 | Very strong importance |
| 9 | Extreme importance |

The matrix must be **positive reciprocal**: $a_{ji} = 1/a_{ij}$.

### Weight Calculation

The **principal eigenvector** of $A$ gives the relative priority (weight) of each criterion. The eigenvector corresponding to the largest eigenvalue $\lambda_{max}$ is normalized to sum to 1.

### Consistency Check

- **Consistency Index**: $CI = \frac{\lambda_{max} - n}{n - 1}$
- **Consistency Ratio**: $CR = CI / RI$, where $RI$ is a tabulated Random Index
- **Rule**: $CR < 0.10$ indicates acceptable consistency in the expert judgments

In [ ]:
def ahp_weights(pairwise_matrix):
    """
    Calculate criteria weights using the Analytic Hierarchy Process.

    Parameters:
        pairwise_matrix (np.ndarray): Square positive reciprocal matrix (Saaty's 1-9 scale).

    Returns:
        weights (np.ndarray): Normalized criteria weights summing to 1.
        cr (float): Consistency Ratio. CR < 0.10 indicates acceptable consistency.
    """
    n = pairwise_matrix.shape[0]

    # Principal eigenvector method
    eigenvalues, eigenvectors = np.linalg.eig(pairwise_matrix)
    max_idx = np.argmax(eigenvalues.real)
    lambda_max = eigenvalues[max_idx].real
    priority_vector = np.abs(eigenvectors[:, max_idx].real)

    # Normalize to obtain weights
    weights = priority_vector / priority_vector.sum()

    # Consistency Index (CI)
    ci = (lambda_max - n) / (n - 1) if n > 1 else 0.0

    # Random Index (RI) table for matrices of size 1-10 (Saaty, 1980)
    ri_table = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
                6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
    ri = ri_table.get(n, 1.49)

    # Consistency Ratio (CR)
    cr = ci / ri if ri > 0 else 0.0

    return weights, cr

## 2. TOPSIS: Ranking Alternatives

### Algorithm Steps

Given a decision matrix $X$ of shape `(n_alternatives, n_criteria)` and weight vector $w$:

1. **Normalize** each column by its L2 norm: $r_{ij} = x_{ij} / \sqrt{\sum_i x_{ij}^2}$
2. **Weight** the normalized matrix: $v_{ij} = w_j \cdot r_{ij}$
3. Identify the **Positive Ideal Solution (PIS)** and **Negative Ideal Solution (NIS)**:
   - For benefit criteria: $PIS_j = \max(v_{ij})$, $NIS_j = \min(v_{ij})$
   - For cost criteria: $PIS_j = \min(v_{ij})$, $NIS_j = \max(v_{ij})$
4. Compute **Euclidean distances** $S_i^+$ (to PIS) and $S_i^-$ (to NIS)
5. Calculate **relative closeness**: $C_i^* = S_i^- / (S_i^+ + S_i^-)$

The alternative with the highest $C^*$ is ranked first (closest to ideal, farthest from anti-ideal).

In [ ]:
def topsis_rank(decision_matrix, weights, benefit_criteria):
    """
    Rank alternatives using the TOPSIS method.

    Parameters:
        decision_matrix (np.ndarray): Shape (n_alternatives, n_criteria).
        weights (np.ndarray): Criteria weights from AHP (length n_criteria).
        benefit_criteria (list of bool): True = benefit (higher better), False = cost.

    Returns:
        relative_closeness (np.ndarray): C* score for each alternative in [0, 1].
        ranking (np.ndarray): Indices sorted by C* descending (best first).
    """
    # Step 1: Vector normalization
    norms = np.sqrt((decision_matrix ** 2).sum(axis=0))
    normalized = decision_matrix / norms

    # Step 2: Weighted normalized matrix
    weighted = normalized * weights

    # Step 3: PIS and NIS
    n_crit = weighted.shape[1]
    pis = np.zeros(n_crit)
    nis = np.zeros(n_crit)
    for j in range(n_crit):
        if benefit_criteria[j]:
            pis[j] = weighted[:, j].max()
            nis[j] = weighted[:, j].min()
        else:
            pis[j] = weighted[:, j].min()
            nis[j] = weighted[:, j].max()

    # Step 4: Distances
    s_plus = np.sqrt(((weighted - pis) ** 2).sum(axis=1))
    s_minus = np.sqrt(((weighted - nis) ** 2).sum(axis=1))

    # Step 5: Relative closeness
    relative_closeness = s_minus / (s_plus + s_minus)

    ranking = np.argsort(relative_closeness)[::-1]
    return relative_closeness, ranking

## 3. Application: Wave Energy Site Selection

### 3.1 Define Criteria and AHP Pairwise Matrix

The pairwise comparison matrix below encodes expert judgment. For example, wave power is rated 3x more important than energy consistency, 5x more than distance to shore, etc.

In [ ]:
criteria_names = [
    "Wave Power (kW/m)",
    "Energy Consistency (%)",
    "Dist. to Shore (km)",
    "Dist. to Grid (km)",
    "Env. Sensitivity (1-5)"
]

# Benefit = True (higher is better), Cost = False (lower is better)
benefit_flags = [True, True, False, False, False]

# AHP Pairwise Comparison Matrix (Saaty's 1-9 scale)
ahp_matrix = np.array([
    [1,     3,     5,     7,     9],      # Wave Power
    [1/3,   1,     3,     5,     7],      # Energy Consistency
    [1/5,   1/3,   1,     3,     5],      # Distance to Shore
    [1/7,   1/5,   1/3,   1,     3],      # Distance to Grid
    [1/9,   1/7,   1/5,   1/3,   1]       # Env. Sensitivity
])

print("Pairwise Comparison Matrix:")
print(np.array2string(ahp_matrix, precision=3, suppress_small=True))

### 3.2 Compute AHP Weights

In [ ]:
weights, cr = ahp_weights(ahp_matrix)

print("=" * 60)
print("AHP CRITERIA WEIGHTING")
print("=" * 60)
for name, w in zip(criteria_names, weights):
    print(f"  {name:30s}: {w:.4f}")
print(f"\n  Consistency Ratio (CR): {cr:.4f}")
if cr < 0.10:
    print("  CR < 0.10 -- Judgments are consistent.")
else:
    print("  CR >= 0.10 -- Revise pairwise comparisons!")

### 3.3 Decision Matrix: 6 Candidate Sites

Each row represents a candidate site, and each column corresponds to one of the five criteria. In a real application, these values would come from wave model output and GIS analysis.

In [ ]:
site_names = ["Site A", "Site B", "Site C", "Site D", "Site E", "Site F"]

decision_matrix = np.array([
    [35.0,  55.0,  15.0,  20.0,  3],    # Site A
    [50.0,  70.0,  25.0,  30.0,  2],    # Site B
    [28.0,  45.0,   8.0,  12.0,  5],    # Site C
    [45.0,  65.0,  18.0,  22.0,  1],    # Site D
    [40.0,  60.0,  12.0,  15.0,  4],    # Site E
    [55.0,  75.0,  30.0,  35.0,  2],    # Site F
])

print("Decision Matrix:")
print(f"{'Site':<10}", end='')
for name in criteria_names:
    print(f"{name[:15]:>16}", end='')
print()
print("-" * 90)
for i, site in enumerate(site_names):
    print(f"{site:<10}", end='')
    for j in range(decision_matrix.shape[1]):
        print(f"{decision_matrix[i, j]:>16.1f}", end='')
    print()

### 3.4 TOPSIS Ranking

In [ ]:
closeness, ranking = topsis_rank(decision_matrix, weights, benefit_flags)

print("=" * 60)
print("TOPSIS RANKING RESULTS")
print("=" * 60)
print(f"  {'Rank':<6} {'Site':<10} {'C*':<10}")
print("  " + "-" * 26)
for rank, idx in enumerate(ranking, 1):
    print(f"  {rank:<6} {site_names[idx]:<10} {closeness[idx]:.4f}")
print(f"\n  Best site: {site_names[ranking[0]]}")

## 4. Interpretation

The TOPSIS $C^*$ score ranges from 0 (worst) to 1 (best). The site with the highest $C^*$ value offers the best trade-off across all weighted criteria.

**Key takeaways:**
- Sites with high wave power and consistency score well on the benefit criteria
- Sites close to shore and grid infrastructure score well on the cost criteria
- The AHP weights ensure that wave resource quality (power, consistency) is prioritized over logistical factors
- The consistency ratio (CR < 0.10) confirms that the expert pairwise judgments are logically coherent

### Next Steps

In the companion notebook `swan_coupled_ahp_topsis.ipynb`, we replace the tabular data above with **SWAN model output**, computing all criteria directly from resolved nearshore wave fields.